<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [1]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    mnist = fetch_openml("mnist_784", as_frame=False)
    X, y = mnist.data, mnist.target
    
    y = y.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed
    )

    return X_train, X_test, y_train, y_test

Random Forest e AdaBoost são baseados em árvores de decisão, que fazem divisões por limiares nos valores dos atributos, não por distâncias. Por isso, a escala dos dados não afeta o resultado do modelo e a normalização não é necessária. Modelos sensíveis à escala, como kNN, exigiriam normalização, mas não é o caso aqui.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [2]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [3]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    
    return acc

Nessa fase, implementei a função de avaliação para avaliar o desempenho dos modelos de Ensemble. A acurácia foi a métrica selecionada, pois mostra a porcentagem total de acertos do modelo ao classificar as imagens de teste do Fashion MNIST. Essa função é fundamental para avaliar qual dos dois, Random Forest ou AdaBoost, é mais eficiente na tarefa de reconhecimento de padrões em imagens.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [4]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed=seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed=seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed=seed)
    else:
        raise ValueError("model_type deve ser 'rf' (Random Forest) ou 'ab' (AdaBoost)")

    accuracy = evaluate(model, X_test, y_test)
    
    return accuracy

**Em qual profundidade começa o overfitting?**
A partir da profundidade 4 ou 5, já é possível notar o overfitting. Observei que, à medida que a profundidade aumenta, a acurácia de treino se aproxima de 100%. Porém, a acurácia de teste para de crescer ou até diminui. Isso sugere que a árvore deixou de aprender padrões gerais e passou a "decorar" o ruído presente nos dados de treinamento.
**Por que a árvore consegue 100% no treino quando max_depth=None?**
A árvore atinge 100% porque, na ausência de um limite de profundidade, ela prossegue com as divisões (galhos) até que cada folha contenha dados de apenas uma classe (folhas puras). Basicamente, ela estabelece uma regra específica para cada pequena variação dos dados de treinamento, resultando em um modelo perfeito para o treinamento, mas ineficaz para classificar novos dados.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [5]:
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = load_data(seed=42)

print("Treinando Random Forest...")
model_rf = train_random_forest(X_train, y_train, seed=42)

print("Treinando AdaBoost...")
model_ab = train_adaboost(X_train, y_train, seed=42)

modelos = {"Random Forest": model_rf, "AdaBoost": model_ab}
resultados = {}

for nome, modelo in modelos.items():
    y_pred = modelo.predict(X_test)
    
    resultados[nome] = {
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão": precision_score(y_test, y_pred, average='weighted'),
        "Recall": recall_score(y_test, y_pred, average='weighted'),
        "F1-Score": f1_score(y_test, y_pred, average='weighted')
    }

import pandas as pd
df_resultados = pd.DataFrame(resultados).T
display(df_resultados)

Treinando Random Forest...
Treinando AdaBoost...


,Acurácia,Precisão,Recall,F1-Score
Random Forest,0.967286,0.967287,0.967286,0.967271
AdaBoost,0.642571,0.685202,0.642571,0.648345


O modelo com melhor desempenho foi o Random Forest, alcançando uma acurácia de cerca de 96,7%, superando consideravelmente o AdaBoost, que registrou aproximadamente 64,2%. Isso se deve ao fato de que o Random Forest é capaz de captar com mais precisão as sutilezas das imagens de roupas por meio de suas diversas árvores de decisão profundas. Na sua configuração padrão, o AdaBoost usa modelos simples que não conseguiram captar os padrões complexos desse conjunto de dados.

# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?
Sim, quando mudamos a seed de 42 para 7, os resultados mudaram um pouco. Isso acontece porque a seed afeta tanto o embaralhamento dos dados na divisão entre treino e teste quanto a seleção de características para cada nó nas árvores do Random Forest.

## Responda:
- O experimento é reprodutível? Justifique.
Sim, o experimento pode ser reproduzido completamente. Um exemplo disso é que as duas execuções com a Seed 42 produziram exatamente o mesmo resultado de acurácia (0.967286). Isso mostra que, ao estabelecer o estado aleatório por meio da seed, asseguramos que qualquer outra pessoa que execute esse código receberá os mesmos resultados, tornando o processo científico e confiável.

**Solução**:

In [6]:
acc_42 = run_pipeline(model_type="rf", seed=42)
print(f"Resultado com Seed 42: {acc_42:.6f}")

acc_42_repete = run_pipeline(model_type="rf", seed=42)
print(f"Repetição com Seed 42: {acc_42_repete:.6f}")

acc_7 = run_pipeline(model_type="rf", seed=7)
print(f"Resultado com Seed 7: {acc_7:.6f}")

print("\nConclusão:")
if acc_42 == acc_42_repete:
    print("O experimento é reprodutível: a mesma seed gera o mesmo resultado.")
else:
    print("O experimento não é reprodutível.")

Resultado com Seed 42: 0.967286
Repetição com Seed 42: 0.967286
Resultado com Seed 7: 0.971714

Conclusão:
O experimento é reprodutível: a mesma seed gera o mesmo resultado.


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
Sim, existe um leve grau de overfitting. Apesar do modelo ter um desempenho excepcional no teste (96,7\%), o fato de ele alcançar quase 100\% no treinamento indica que ele se adaptou perfeitamente aos dados de exemplo. Quando a acurácia de treino é muito maior do que a de teste, isso é chamado de overfitting.
- Qual modelo tende a sofrer mais com isso?
O modelo que geralmente apresenta maior sensibilidade ao overfitting é o Random Forest (ou árvores de decisão individuais sem poda). Isso ocorre porque o Random Forest utiliza árvores muito profundas com o objetivo de minimizar o erro de treinamento a zero. Por outro lado, o AdaBoost tende a apresentar menos problemas com overfitting "bruto", uma vez que emprega modelos bastante simples (stumps/árvores rasas), o que torna mais difícil a memorização de ruídos nos dados de treinamento. No entanto, se não for adequadamente ajustado, pode enfrentar mais dificuldades para reconhecer padrões complexos.

In [7]:
train_acc_rf = model_rf.score(X_train, y_train)
test_acc_rf = model_rf.score(X_test, y_test)

print(f"Random Forest - Acurácia Treino: {train_acc_rf:.4f}")
print(f"Random Forest - Acurácia Teste: {test_acc_rf:.4f}")

gap = train_acc_rf - test_acc_rf
print(f"Diferença (Gap): {gap:.4f}")

Random Forest - Acurácia Treino: 1.0000
Random Forest - Acurácia Teste: 0.9673
Diferença (Gap): 0.0327


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

No Random Forest, o desempenho não mudou de forma drástica, mesmo reduzindo o número de árvores para apenas 10, o modelo ainda manteve uma acurácia robusta acima de 94%. No AdaBoost, o aumento de estimadores (de 50 para 100) trouxe uma melhora perceptível (subindo de ~64% para ~71%), mas o modelo ainda parece limitado pela simplicidade dos seus componentes básicos.

## Responda:
- Qual modelo é mais sensível a mudanças?

O AdaBoost é o modelo mais sensível. Isso ocorre porque, sendo um algoritmo de Boosting, ele depende inteiramente da correção sequencial dos erros. Cada novo estimador adicionado tem um impacto direto na tentativa de ajustar o modelo aos dados difíceis. O Random Forest, por outro lado, é extremamente resiliente: como ele baseia-se na média de árvores independentes (Bagging), ele atinge um desempenho alto muito rápido e não sofre grandes perdas mesmo quando diminuímos o número de árvores significativamente.

In [9]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

model_rf_low = RandomForestClassifier(n_estimators=10, random_state=42, n_jobs=-1)
model_rf_low.fit(X_train, y_train)
acc_rf_low = model_rf_low.score(X_test, y_test)

model_ab_high = AdaBoostClassifier(n_estimators=100, random_state=42)
model_ab_high.fit(X_train, y_train)
acc_ab_high = model_ab_high.score(X_test, y_test)

print(f"Random Forest (10 árvores): {acc_rf_low:.4f}")
print(f"AdaBoost (100 modelos): {acc_ab_high:.4f}")

Random Forest (10 árvores): 0.9456
AdaBoost (100 modelos): 0.7174


# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?

Não, a acurácia isoladamente pode levar a conclusões incorretas, principalmente em situações em que as classes podem estar desbalanceadas (embora o Fashion MNIST seja bastante equilibrado). Ela não sabe se o modelo comete muitos erros em uma classe particular (por exemplo, confundindo constantemente camisas com camisetas). Métricas como Precisão, Recall e F1-Score (calculadas na Q5) são essenciais, pois refletem o equilíbrio entre falsos positivos e falsos negativos, proporcionando uma visão com mais detalhe da performance real por categoria.

2. Como você garante que o resultado não ocorreu por acaso?

Garantimos isso por meio da Reprodutibilidade e da utilização de diversas avaliações. Na Questão 6, utilizamos o random_state (seed) para fixar a aleatoriedade e garantir que o resultado seja consistente. Para garantir ainda mais confiabilidade, é recomendável o uso da Validação Cruzada (Cross-Validation), na qual o modelo é treinado e avaliado em várias divisões distintas dos dados. Isso assegura que a alta acurácia não seja resultado de uma "sorte" em uma divisão específica entre treino e teste.

3. Cite dois possíveis problemas metodológicos neste experimento.

O primeiro ponto é que a gente não fez um ajuste de hiperparâmetros de forma mais cuidadosa. Acabamos comparando o AdaBoost “do jeito padrão” com o Random Forest, mas provavelmente o AdaBoost teria um desempenho melhor se tivéssemos ajustado, por exemplo, a profundidade do modelo base.
Outro ponto é que não analisamos a matriz de confusão. Isso faria bastante diferença, porque ajudaria a entender melhor quais peças de roupa o modelo está confundindo, algo bem comum quando a gente olha só para métricas gerais de acurácia.

4. O pipeline implementado é confiável? Justifique.

O pipeline está bem organizado do ponto de vista técnico, já que separa direitinho as etapas de carregamento, pré-processamento, treino e avaliação.
Por outro lado, para dizer que ele é realmente robusto para produção, ainda faltam algumas coisas. Por exemplo, incluir uma etapa de normalização/escalonamento dos pixels e também fazer uma validação estatística mais rigorosa, como um teste t de Student, para ter mais segurança ao afirmar que o Random Forest realmente performa melhor que o AdaBoost nesse cenário.

In [ ]:
# TODO: implemente load_data